# UD5.06. Convolución y redes convolucionales

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 13 y 14 de los apuntes · Criterios **2.c** y **2.d**

---

En el cuaderno `UD5_04` una red neuronal perdió contra una regla de una línea. Este
cuaderno es la otra cara de la misma moneda, y contesta la misma pregunta del criterio 2.c
con la respuesta contraria.

La diferencia está en una frase:

> **En TechStore las características las construyó la UD3 a mano. En una imagen no hay
> ninguna característica construida, y construirla es el problema.**

Aquí se ve por qué una red densa no puede con una imagen, qué hace una convolución, y
cuánto gana. Y se hace en el orden correcto: primero la convolución a mano con NumPy, con
filtros escritos por nosotros, y solo después la capa de Keras.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras

SEMILLA = 20262027
keras.utils.set_random_seed(SEMILLA)

CLASES = ["camiseta", "pantalon", "jersey", "vestido", "abrigo",
          "sandalia", "camisa", "zapatilla", "bolso", "botin"]

(X_todo, y_todo), (X_p, y_p) = keras.datasets.fashion_mnist.load_data()
X_ent = X_todo[:10000].astype("float32") / 255.0
y_ent = y_todo[:10000]
X_val = X_todo[10000:12000].astype("float32") / 255.0
y_val = y_todo[10000:12000]
X_pru = X_p.astype("float32") / 255.0
y_pru = y_p

print(f"entrenamiento {X_ent.shape}   validacion {X_val.shape}   prueba {X_pru.shape}")

---

## 1. Las tres razones por las que una densa no sirve

### 1.1. Explosión de parámetros

In [ ]:
def parametros_densa(n_entradas, n_unidades):
    return n_unidades * (n_entradas + 1)


def parametros_conv(k, canales_entrada, canales_salida):
    return canales_salida * (k * k * canales_entrada + 1)


print(f"{'imagen':>18} {'entradas':>10} {'densa de 1000':>16} "
      f"{'conv 32 de 3x3':>16} {'razon':>10}")
print("-" * 76)
for alto, ancho, canales in [(28, 28, 1), (32, 32, 3), (150, 150, 3), (224, 224, 3)]:
    entradas = alto * ancho * canales
    d = parametros_densa(entradas, 1000)
    c = parametros_conv(3, canales, 32)
    print(f"{f'{alto}x{ancho}x{canales}':>18} {entradas:>10,} {d:>16,} {c:>16,} "
          f"{d / c:>10,.0f}")
print()
print("Fijate en la columna de la convolucion: NO cambia con el tamaño de la imagen.")
print("Solo depende del tamaño del filtro y del numero de canales. Esa independencia")
print("es la propiedad clave, y es lo que hace que la misma arquitectura sirva")
print("para imagenes de 32 y de 224 pixeles.")

### 1.2. `Flatten` tira la estructura espacial

Esta es la más difícil de ver y la más importante. Vamos a hacerla visible: si se
**baraja el orden de los píxeles** —siempre igual, con la misma permutación para todas las
imágenes— una red densa aprende exactamente lo mismo, porque para ella el píxel de al lado
y el de doscientas posiciones más allá son igual de vecinos.

In [ ]:
rng = np.random.default_rng(SEMILLA)
permutacion = rng.permutation(28 * 28)

def baraja(X):
    return X.reshape(len(X), -1)[:, permutacion].reshape(-1, 28, 28)


X_ent_b, X_val_b, X_pru_b = baraja(X_ent), baraja(X_val), baraja(X_pru)

fig, ejes = plt.subplots(1, 4, figsize=(10, 2.8))
for eje, (titulo, imagen) in zip(ejes, [
        ("original", X_ent[0]), ("barajada", X_ent_b[0]),
        ("original", X_ent[3]), ("barajada", X_ent_b[3])]):
    eje.imshow(imagen, cmap="gray")
    eje.set_title(titulo, fontsize=9)
    eje.axis("off")
fig.suptitle("Para un ojo humano la version barajada no es nada. Veamos si para "
             "una red densa lo es", y=1.05)
fig.tight_layout()
plt.show()

In [ ]:
def densa():
    return keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])


resultados = []
for etiqueta, (a, b, c) in [("imagenes normales", (X_ent, X_val, X_pru)),
                            ("pixeles barajados", (X_ent_b, X_val_b, X_pru_b))]:
    keras.utils.set_random_seed(SEMILLA)
    m = densa()
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    m.fit(a, y_ent, epochs=8, batch_size=64, validation_data=(b, y_val), verbose=0)
    _, acc = m.evaluate(c, y_pru, verbose=0)
    resultados.append((etiqueta, acc))
    print(f"red densa, {etiqueta:20} exactitud de prueba {acc:.4f}")

print()
print(f"Diferencia: {abs(resultados[0][1] - resultados[1][1]):.4f}")
print()
print("Practicamente la misma. La red densa NUNCA supo que las imagenes tenian")
print("estructura espacial: la tiro en el Flatten, antes de la primera capa.")

> **Esa es la demostración.** Una red densa da igual resultado con los píxeles en su sitio
> que con los píxeles barajados, porque nunca usó la información de que estaban en su
> sitio. Una convolucional no: al final del cuaderno se repite el experimento con ella y sí
> pierde, y cuánto pierde es la medida de cuánta información espacial estaba usando.

---

## 2. Una convolución, a mano

Un filtro es una matriz pequeña de pesos. Se coloca sobre una zona de la imagen, se
multiplica elemento a elemento, se suman los productos, y ese número va a la salida.
Después se desplaza y se repite.

In [ ]:
def convolucion_2d(imagen, filtro):
    # La convolucion entera, sin bibliotecas. Relleno 'valid' y paso 1.
    k = filtro.shape[0]
    alto, ancho = imagen.shape[0] - k + 1, imagen.shape[1] - k + 1
    salida = np.zeros((alto, ancho))
    for i in range(alto):
        for j in range(ancho):
            ventana = imagen[i:i + k, j:j + k]
            salida[i, j] = (ventana * filtro).sum()
    return salida


# Filtros clasicos de vision por computador, escritos a mano.
FILTROS = {
    "bordes verticales (Sobel)": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    "bordes horizontales":       np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float),
    "desenfoque":                np.ones((3, 3)) / 9,
    "realce":                    np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], float),
}

imagen = X_ent[3]
fig, ejes = plt.subplots(1, 5, figsize=(13, 3))
ejes[0].imshow(imagen, cmap="gray")
ejes[0].set_title(f"original\n({CLASES[y_ent[3]]})", fontsize=9)
for eje, (nombre, filtro) in zip(ejes[1:], FILTROS.items()):
    eje.imshow(convolucion_2d(imagen, filtro), cmap="gray")
    eje.set_title(nombre, fontsize=8)
for eje in ejes:
    eje.axis("off")
fig.suptitle("El mismo filtro de nueve numeros, aplicado en TODA la imagen", y=1.06)
fig.tight_layout()
plt.show()

Fíjate en el detector de bordes verticales: responde fuerte en los laterales de la prenda
y casi nada en el resto. **Eso es exactamente lo que aprende la primera capa de una red
convolucional**, solo que en vez de escribirlo nosotros, lo aprende ella.

### Comprobación contra Keras

In [ ]:
filtro = FILTROS["bordes verticales (Sobel)"]

capa = keras.layers.Conv2D(1, 3, padding="valid", use_bias=False)
capa.build((None, 28, 28, 1))
# Keras espera el filtro con forma (k, k, canales_entrada, canales_salida).
capa.set_weights([filtro[:, :, np.newaxis, np.newaxis]])

mio = convolucion_2d(imagen, filtro)
suyo = capa(imagen[np.newaxis, ..., np.newaxis]).numpy()[0, ..., 0]

print("a mano:", mio.shape, "  Keras:", suyo.shape)
print("diferencia maxima:", np.abs(mio - suyo).max())
assert np.allclose(mio, suyo, atol=1e-4)
print()
print("Identicas. Conv2D hace EXACTAMENTE lo que acabas de escribir, con")
print("la salvedad de que sus filtros se aprenden en vez de escribirse.")

---

## 3. Los tres parámetros, y la fórmula del tamaño de salida

$$\text{salida} = \left\lfloor \frac{\text{entrada} + 2p - k}{s} \right\rfloor + 1$$

Hay que saber aplicarla sin ejecutar nada, porque es lo que permite diseñar una
arquitectura en un papel.

In [ ]:
def tamano_salida(entrada, k, relleno, paso):
    p = (k - 1) // 2 if relleno == "same" else 0
    return (entrada + 2 * p - k) // paso + 1


casos = [(28, 3, "valid", 1), (28, 3, "same", 1), (28, 5, "valid", 1),
         (28, 5, "same", 1), (32, 3, "same", 2), (224, 7, "same", 2)]

print(f"{'entrada':>8} {'k':>3} {'relleno':>8} {'paso':>5} {'formula':>8} {'Keras':>7}")
print("-" * 46)
for entrada, k, relleno, paso in casos:
    formula = tamano_salida(entrada, k, relleno, paso)
    capa = keras.layers.Conv2D(1, k, strides=paso, padding=relleno)
    real = capa(np.zeros((1, entrada, entrada, 1), dtype="float32")).shape[1]
    print(f"{entrada:>8} {k:>3} {relleno:>8} {paso:>5} {formula:>8} {real:>7}")
print()
print("La formula y Keras coinciden en los seis casos.")

### Dos filtros de 3×3 ven lo mismo que uno de 5×5

Y ese es el motivo de que casi todo sea 3×3 desde VGG. El **campo receptivo** —la zona del
original que ve cada neurona— crece igual, pero con menos parámetros y con una no
linealidad de más en medio.

In [ ]:
canales = 32
uno_grande = parametros_conv(5, canales, canales)
dos_pequenos = 2 * parametros_conv(3, canales, canales)

print(f"Una capa de 5x5, {canales} -> {canales} canales: {uno_grande:>7,} parametros")
print(f"Dos capas de 3x3 seguidas:                 {dos_pequenos:>7,} parametros")
print(f"Ahorro: {(1 - dos_pequenos / uno_grande) * 100:.0f} %")
print()
print("Y las dos ven una zona de 5x5 del original. Ademas, entre las dos capas de")
print("3x3 hay una ReLU que la de 5x5 no tiene: mas capacidad expresiva, no menos.")

---

## 4. La agrupación

`MaxPooling2D((2,2))` se queda con el máximo de cada cuadrado de 2×2. No tiene parámetros:
no se aprende nada. Reduce el cómputo, aumenta el campo receptivo y da algo de tolerancia a
desplazamientos pequeños.

In [ ]:
mapa = convolucion_2d(imagen, FILTROS["bordes verticales (Sobel)"])
entrada = np.abs(mapa)[np.newaxis, ..., np.newaxis].astype("float32")

maximo = keras.layers.MaxPooling2D(2)(entrada).numpy()[0, ..., 0]
promedio = keras.layers.AveragePooling2D(2)(entrada).numpy()[0, ..., 0]
global_ = keras.layers.GlobalAveragePooling2D()(entrada).numpy()

fig, ejes = plt.subplots(1, 3, figsize=(10, 3.2))
for eje, (titulo, m) in zip(ejes, [
        (f"mapa de rasgos {entrada.shape[1]}x{entrada.shape[2]}", np.abs(mapa)),
        (f"MaxPooling 2x2 -> {maximo.shape[0]}x{maximo.shape[1]}", maximo),
        (f"AveragePooling 2x2 -> {promedio.shape[0]}x{promedio.shape[1]}", promedio)]):
    eje.imshow(m, cmap="gray")
    eje.set_title(titulo, fontsize=9)
    eje.axis("off")
fig.suptitle("MaxPooling se queda con el rasgo mas fuerte; el promedio lo difumina",
             y=1.05)
fig.tight_layout()
plt.show()

print(f"GlobalAveragePooling2D reduce {entrada.shape[1:]} a {global_.shape[1:]}:")
print("un numero por canal. Eso es lo que sustituye al Flatten en las CNN de ahora.")

In [ ]:
# Por que GlobalAveragePooling y no Flatten: la cuenta.
mapa_final = (7, 7, 512)
print(f"Con un mapa final de {mapa_final}:")
print(f"  Flatten                 -> {np.prod(mapa_final):,} numeros")
print(f"  GlobalAveragePooling2D  -> {mapa_final[2]:,} numeros")
print()
print("Y si detras hay una Dense de 256 unidades:")
print(f"  tras Flatten:  {parametros_densa(int(np.prod(mapa_final)), 256):>10,} parametros")
print(f"  tras GAP:      {parametros_densa(mapa_final[2], 256):>10,} parametros")
print()
print("Flatten vuelve a meter el problema del apartado 1.1 por la puerta de atras.")

---

## 5. La primera red convolucional

El patrón: un extractor de rasgos —bloques convolucionales— y una cabeza que decide.

> **A medida que se avanza, el mapa se hace más pequeño y más profundo.** El alto y el
> ancho se dividen por dos en cada agrupación; el número de filtros se duplica.

In [ ]:
def cnn(cabeza="flatten"):
    capas = [
        keras.layers.Input(shape=(28, 28, 1)),

        # Extractor
        keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        keras.layers.MaxPooling2D(),                       # 28 -> 14
        keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        keras.layers.MaxPooling2D(),                       # 14 -> 7
    ]
    # Cabeza
    capas.append(keras.layers.Flatten() if cabeza == "flatten"
                 else keras.layers.GlobalAveragePooling2D())
    capas += [keras.layers.Dropout(0.3),
              keras.layers.Dense(10, activation="softmax")]
    return keras.Sequential(capas)


modelo_cnn = cnn()
modelo_cnn.summary()

### La comparación, con el conjunto completo

Y aquí hay que tener cuidado con el régimen, que es la lección de la P5.1. **Con 10.000
muestras las dos redes empatan**, porque Fashion-MNIST es pequeño y sencillo y una red
densa llega lejos. La convolución empieza a pagar con el conjunto entero, así que la
comparación se hace con las 60.000 imágenes.

Son unos dos minutos de CPU en total. Si vas con prisa, baja `EPOCAS` a 4 y apunta que lo
has hecho.

In [ ]:
# Las dos redes, con el MISMO presupuesto de epocas y el conjunto completo.
X_ent_c = X_ent[..., np.newaxis]
X_val_c = X_val[..., np.newaxis]
X_pru_c = X_pru[..., np.newaxis]

X_completo = X_todo.astype("float32") / 255.0
y_completo = y_todo
EPOCAS = 8

filas = []
for nombre, construye, canal in [("densa 128", densa, False),
                                 ("convolucional 32-64", cnn, True)]:
    keras.utils.set_random_seed(SEMILLA)
    m = construye()
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    entrada = X_completo[..., np.newaxis] if canal else X_completo
    prueba = X_pru_c if canal else X_pru
    t0 = time.perf_counter()
    h = m.fit(entrada, y_completo, epochs=EPOCAS, batch_size=64,
              validation_split=0.1, verbose=0)
    segundos = time.perf_counter() - t0
    _, acc = m.evaluate(prueba, y_pru, verbose=0)
    filas.append({"modelo": nombre, "parametros": m.count_params(),
                  "exactitud prueba": acc, "segundos": segundos,
                  "s/epoca": segundos / EPOCAS})
    if canal:
        modelo_cnn, historia_cnn = m, h

print(pd.DataFrame(filas).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
densa_acc = filas[0]["exactitud prueba"]
conv_acc = filas[1]["exactitud prueba"]
print(f"La convolucional gana por {(conv_acc - densa_acc) * 100:+.1f} puntos")
print(f"con {filas[1]['parametros'] / filas[0]['parametros']:.2f} veces los parametros")
print(f"y {filas[1]['segundos'] / filas[0]['segundos']:.1f} veces el tiempo.")

> **La convolucional gana con la mitad de parámetros.** Esa es la frase entera: no es que
> sea más grande, es que su forma encaja con la forma del problema.
>
> Y cuesta unas cuatro veces más tiempo, porque una convolución hace muchas más operaciones
> con muchos menos pesos. Ese intercambio —menos memoria a cambio de más cálculo— es la
> otra mitad de la caracterización, y es contenido del criterio 2.b.

### Un aviso sobre la cabeza, y es un resultado medido

El apartado 4 dijo que `GlobalAveragePooling2D` sustituye al `Flatten` en las CNN de ahora.
Es cierto **y depende de cuántos canales haya**. Vamos a comprobarlo, porque es el tipo de
regla que se copia sin medir.

In [ ]:
filas_cabeza = []
for cabeza in ("flatten", "gap"):
    keras.utils.set_random_seed(SEMILLA)
    m = cnn(cabeza)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(X_completo[..., np.newaxis], y_completo, epochs=EPOCAS, batch_size=64,
          validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_pru_c, y_pru, verbose=0)
    # Cuantos numeros recibe la capa densa final.
    entrada_densa = 7 * 7 * 64 if cabeza == "flatten" else 64
    filas_cabeza.append({"cabeza": cabeza, "parametros": m.count_params(),
                         "entradas de la densa final": entrada_densa,
                         "exactitud prueba": acc})

print(pd.DataFrame(filas_cabeza).to_string(index=False,
                                           float_format=lambda v: f"{v:.4f}"))
print()
print("A esta escala GANA Flatten, y el motivo esta en la tercera columna: con solo")
print("64 canales, GlobalAveragePooling2D deja 64 numeros para distinguir 10 clases.")
print("Es un cuello de botella demasiado estrecho.")
print()
print("GAP gana cuando hay MUCHOS canales: en el cuaderno UD5_07, la base preentrenada")
print("entrega un mapa de 3x3x1280, y ahi Flatten daria 11.520 entradas y millones de")
print("parametros en la densa siguiente.")
print()
print("La regla correcta no es 'GAP siempre', sino: GAP cuando el numero de canales")
print("es holgado frente al numero de clases, y Flatten cuando el mapa final es")
print("pequeño y estrecho. Y se comprueba midiendo, como acabamos de hacer.")

### La prueba de los píxeles barajados, ahora con la convolucional

Con 10.000 muestras otra vez, que para este experimento basta: lo que se compara es la
**caída**, no el valor absoluto.

In [ ]:
def cnn_sobre(X_e, X_v, X_p):
    keras.utils.set_random_seed(SEMILLA)
    m = cnn()
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(X_e[..., np.newaxis], y_ent, epochs=8, batch_size=64,
          validation_data=(X_v[..., np.newaxis], y_val), verbose=0)
    return m.evaluate(X_p[..., np.newaxis], y_pru, verbose=0)[1]


acc_normal = cnn_sobre(X_ent, X_val, X_pru)
acc_barajada = cnn_sobre(X_ent_b, X_val_b, X_pru_b)

print(f"{'':26} {'normales':>10} {'barajados':>11} {'caida':>8}")
print("-" * 58)
print(f"{'red densa':26} {resultados[0][1]:>10.4f} {resultados[1][1]:>11.4f} "
      f"{resultados[0][1] - resultados[1][1]:>8.4f}")
print(f"{'red convolucional':26} {acc_normal:>10.4f} {acc_barajada:>11.4f} "
      f"{acc_normal - acc_barajada:>8.4f}")
print()
print("La densa no se entera de que le han barajado los pixeles: su caida es")
print("negativa, es decir, ruido. La convolucional pierde varios puntos, y esa")
print("caida ES la medida de cuanta informacion espacial estaba usando.")
print()
print("La caida no es enorme, y conviene entender por que: la cabeza Flatten de")
print("esta red recibe 3.136 numeros y funciona como una densa grande, asi que")
print("compensa una parte de lo que las convoluciones han dejado de aportar.")
print("Con la cabeza GlobalAveragePooling2D del apartado anterior, que no tiene")
print("esa holgura, la caida es del triple. El experimento se puede repetir con")
print("cnn('gap') para comprobarlo.")

---

## 6. Qué ve la red

Los filtros de la primera capa y los mapas de activación. No es una curiosidad: es la
forma de comprobar que el extractor ha aprendido algo y no ruido.

In [ ]:
pesos = modelo_cnn.layers[0].get_weights()[0]     # (3, 3, 1, 32)
print("Forma de los filtros de la primera capa:", pesos.shape)

fig, ejes = plt.subplots(4, 8, figsize=(10, 5))
for i, eje in enumerate(ejes.ravel()):
    eje.imshow(pesos[:, :, 0, i], cmap="gray")
    eje.axis("off")
fig.suptitle("Los 32 filtros de 3x3 aprendidos por la primera capa", y=0.97)
fig.tight_layout()
plt.show()

In [ ]:
# Los mapas de activacion: que responde en cada capa ante una imagen concreta.
capas_conv = [c for c in modelo_cnn.layers if isinstance(c, keras.layers.Conv2D)]
extractor = keras.Model(modelo_cnn.inputs, [c.output for c in capas_conv])

muestra = X_pru_c[12][np.newaxis]
mapas = extractor.predict(muestra, verbose=0)

fig = plt.figure(figsize=(12, 5))
eje = fig.add_subplot(len(mapas) + 1, 9, 1)
eje.imshow(muestra[0, ..., 0], cmap="gray")
eje.set_title(f"entrada\n({CLASES[y_pru[12]]})", fontsize=8)
eje.axis("off")

for fila, mapa in enumerate(mapas):
    for col in range(8):
        eje = fig.add_subplot(len(mapas) + 1, 9, (fila + 1) * 9 + col + 2)
        eje.imshow(mapa[0, :, :, col], cmap="viridis")
        eje.axis("off")
        if col == 0:
            eje.set_ylabel(f"capa {fila + 1}")
    fig.text(0.02, 0.62 - fila * 0.3, f"capa {fila + 1}\n{mapa.shape[1:]}", fontsize=8)

fig.suptitle("Ocho de los mapas de rasgos de cada capa convolucional:\n"
             "la primera responde a bordes, la segunda a combinaciones de bordes", y=1.0)
fig.tight_layout()
plt.show()

---

## 7. Dónde se equivoca

La evaluación completa es la de la UD4 y aquí se aplica igual: la matriz de confusión
dice **qué** confunde con qué, y en Fashion-MNIST el resultado tiene sentido de negocio.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred = modelo_cnn.predict(X_pru_c, verbose=0).argmax(axis=1)
mc = confusion_matrix(y_pru, pred, normalize="true")

fig, eje = plt.subplots(figsize=(7.5, 6.5))
imagen_mc = eje.imshow(mc, cmap="Blues", vmin=0, vmax=1)
eje.set_xticks(range(10), CLASES, rotation=45, ha="right", fontsize=8)
eje.set_yticks(range(10), CLASES, fontsize=8)
for i in range(10):
    for j in range(10):
        if mc[i, j] >= 0.02:
            eje.text(j, i, f"{mc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                     color="white" if mc[i, j] > 0.5 else "black")
eje.set_xlabel("lo que dice el modelo");  eje.set_ylabel("lo que es")
eje.set_title("Normalizada por fila: de cada prenda real,\nque proporcion va a cada casilla")
fig.colorbar(imagen_mc, ax=eje, fraction=0.046)
fig.tight_layout()
plt.show()

print(classification_report(y_pru, pred, target_names=CLASES, digits=3))

In [ ]:
# Las confusiones mas frecuentes, ordenadas.
confusiones = [(CLASES[i], CLASES[j], mc[i, j])
               for i in range(10) for j in range(10) if i != j]
confusiones.sort(key=lambda t: -t[2])

print("Las seis confusiones mas frecuentes:")
for real, dicho, valor in confusiones[:6]:
    print(f"  {real:10} -> {dicho:10} {valor:.3f}")
print()
print("Camisa, camiseta, jersey y abrigo son prendas de torso en escala de grises")
print("y 28x28 pixeles. Que se confundan entre si no es un fallo del modelo:")
print("es el limite del conjunto de datos, y decirlo es parte de la evaluacion.")

---

## Ejercicios

### Ejercicio 1. El filtro a mano

Escribe un filtro de 3×3 que detecte bordes **diagonales** y aplícalo con tu
`convolucion_2d`. Después comprueba que `Conv2D` da lo mismo. ¿Qué tendrías que cambiar
para que detecte la otra diagonal?

### Ejercicio 2. La fórmula

Sin ejecutar nada, calcula la forma de salida de cada capa de esta red, y el número de
parámetros de cada una:

```python
keras.Sequential([
    keras.layers.Input(shape=(64, 64, 3)),
    keras.layers.Conv2D(16, 5, padding="valid"),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 3, padding="same", strides=2),
    keras.layers.MaxPooling2D(),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(4),
])
```

Después comprueba con `summary()`. Si no coincide, el error está casi siempre en el
`strides=2`.

### Ejercicio 3. `valid` contra `same`

Construye dos redes idénticas salvo el relleno, con cinco capas convolucionales de 3×3
sobre entradas de 28×28. Con `valid`, ¿cuántas capas caben antes de que el mapa se quede
sin tamaño? Entrena las dos y compara. ¿Compensa la diferencia de exactitud la diferencia
de parámetros?

### Ejercicio 4. Max contra promedio

Sustituye los `MaxPooling2D` por `AveragePooling2D` y compara exactitud y curvas. Escribe
dos frases sobre por qué el máximo suele ganar en clasificación, y un caso en el que el
promedio sería preferible.

### Ejercicio 5. `Flatten` contra `GlobalAveragePooling2D`

Cambia la cabeza de la CNN por `Flatten` + `Dense(128)` + `Dense(10)` y compara: número de
parámetros, exactitud de prueba y hueco entre entrenamiento y validación. Relaciona el
resultado con el apartado 1.1.

### Ejercicio 6. El límite del conjunto

Coge las veinte imágenes de prueba en las que el modelo se equivoca con más confianza y
dibújalas con su etiqueta real y la predicha. ¿Cuántas dirías **tú** correctamente?
Escribe una frase sobre qué parte del error es del modelo y qué parte del conjunto de
datos. Es la misma idea que los límites declarados de la UD4.

---

## Lo que hay que llevarse de aquí

1. **Los parámetros de una convolución no dependen del tamaño de la imagen.** Los de una
   densa, sí, y por eso se disparan.
2. **`Flatten` tira la estructura espacial**, y se demuestra barajando los píxeles: la
   densa no se entera y la convolucional se hunde.
3. **Una convolución es un detector de patrones locales aplicado en todas partes**, y es
   literalmente lo que escribiste con NumPy.
4. **La fórmula del tamaño de salida se aplica en un papel**, y hay que saber hacerlo.
5. **Dos capas de 3×3 ven lo mismo que una de 5×5**, con menos parámetros y una no
   linealidad más.
6. **La agrupación no tiene parámetros.** Reduce, amplía el campo receptivo y tolera
   desplazamientos.
7. **`GlobalAveragePooling2D` en lugar de `Flatten` cuando hay muchos canales**, o la
   primera densa vuelve a tener millones de parámetros. Con pocos canales es al revés, y
   está medido en este cuaderno: la regla se comprueba, no se copia.
8. **La convolucional gana con la mitad de parámetros**, y cuesta cuatro veces más tiempo:
   no es más grande, es que su forma encaja con la del problema.
9. **El régimen manda**: con 10.000 muestras las dos redes empatan y con 60.000 no. Una
   comparación sin decir en qué régimen se hizo vale poco.
10. **Los filtros aprendidos se pueden dibujar**, y es la comprobación de que el extractor
    ha aprendido algo.
11. **Parte del error es del conjunto de datos, no del modelo**, y hay que decirlo.